# 05 · Supervised fine-tuning

**The decision this notebook supports:** can I teach the model the right *behaviour*, meaning format, tone and length, even where I cannot teach it facts?

Brisk again, with one thing worth care: **assistant-only loss masking**. SFT differs from CPT in exactly one respect that matters. We do not want the model to learn to predict the user's question. We only want it to learn to produce the assistant's answer. So the question tokens are masked out of the loss with `-100`.

Getting this wrong is the most common silent SFT bug. The loss still falls, the model still trains, and it learns partly to generate questions. Inspect the mask.

In [ ]:
# Load this project's shared paths, configuration, scoring, and file helpers.
# sys controls Python's import search path; adding '..' assumes the notebook folder
# is the working directory. Import the project root, not src itself, so src/secrets.py
# does not replace Python's own secrets module. warnings controls diagnostic messages;
# the existing filter hides warnings, which can also hide useful compatibility clues.
# json handles JSON data; it is imported here for later cells (unused in this setup).
# CFG is a nested dictionary: CFG["model"]["name"] reads its model-name setting.
import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])

In [ ]:
# Load the tokenizer (text <-> token IDs) and the pretrained next-token model.
# transformers supplies AutoTokenizer and AutoModelForCausalLM: the Auto classes
# choose the matching implementation from the saved checkpoint configuration.
# torch supplies tensors and GPU execution. float32 uses four bytes per weight;
# .to(DEVICE) moves the weights to Apple's MPS backend, explicitly selected here.
# This loads the original base; it does not resume the earlier CPT adapter.
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
MODEL_PATH, DEVICE = CFG["model"]["base_path"], lab.device()
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
# Reuse EOS for padding only when no pad token exists. The collator will mask
# padding positions separately so an actual answer-ending EOS can still be learned.
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.float32).to(DEVICE)
print("loaded:", sum(p.numel() for p in model.parameters())/1e6, "M params on", DEVICE)

In [ ]:
# Read the SFT JSON arrays already present on disk: one training list and one
# validation list. json.loads parses text into Python lists/dictionaries; dumps
# formats the first row for inspection. No data is generated in this cell.
# Notebook05 uses question and answer fields; additional provenance metadata is
# retained in the file but ignored by encode(). These paths may hold session data.
import json
train = json.loads((lab.GEN/"sft_train.json").read_text())
val   = json.loads((lab.GEN/"sft_val.json").read_text())
print(len(train), "train |", len(val), "val")
print(json.dumps(train[0], indent=2))

## Build the masked example, and look at it

In [ ]:
# Create chat-formatted supervised examples with assistant-only loss targets.
# torch supplies the tensor tools used by later cells; this encoder itself returns
# Python lists. SFT here changes the example format and desired responses as well
# as the loss mask; masking alone is not the complete distinction from raw-text CPT.
import torch
SYSTEM = ("You are the customer assistant for Meridian Motors. Answer only from "
          "Meridian's official documents. If you do not have the information, say so plainly.")

# The default max_len is read when this function is defined, not on every call.
# Render system+question with the assistant-start marker, then append the desired
# answer and an EOS ending. The fixed SYSTEM text is part of every training prompt.
def encode(ex, max_len=CFG["training"]["sft"]["max_seq_len"]):
    prompt = tok.apply_chat_template(
        [{"role":"system","content":SYSTEM},{"role":"user","content":ex["question"]}],
        tokenize=False, add_generation_prompt=True)
    full = prompt + ex["answer"] + tok.eos_token
    # Tokenize prompt and full sequence without adding another set of special tokens.
    # The full-ID slice [:max_len] silently truncates long examples; if the prompt alone
    # fills the limit, no answer target remains. Inspect lengths before new-data training.
    # Masking assumes separately tokenized prompt IDs match the full sequence's prefix;
    # that boundary should be checked when changing the tokenizer/template.
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full,  add_special_tokens=False)["input_ids"][:max_len]
    # Copy the IDs before replacing prompt labels, so input_ids still contains the
    # full question context. -100 is cross-entropy's ignore value, not a vocabulary ID.
    # Prompt tokens remain visible to attention but are not direct loss targets.
    labels = list(f_ids)
    # range visits prompt positions up to the available truncated sequence length.
    # The causal LM shifts logits and labels internally: the final prompt position
    # predicts the first answer token. Do not manually shift these lists again.
    for i in range(min(len(p_ids), len(labels))):
        labels[i] = -100                      # <- the mask
    # [1]*length repeats a list entry: all positions are currently real tokens.
    # Padding will be added by collate(), with attention 0 and labels -100.
    return {"input_ids": f_ids, "attention_mask":[1]*len(f_ids), "labels": labels}

# Inspect one encoded row before training. Generator expressions count ignored
# and supervised labels. The filtered decode shows the answer plus its EOS control
# token (repr makes special characters visible), rather than the masked question.
e = encode(train[0])
print("total tokens :", len(e["input_ids"]))
print("masked (-100):", sum(1 for x in e["labels"] if x == -100))
print("learned from :", sum(1 for x in e["labels"] if x != -100))
print("\nthe part the model is scored on:")
print(repr(tok.decode([x for x in e["labels"] if x != -100])))

That last line is the proof. It should contain **only the assistant's answer**. If the question text appears there, the mask is wrong and everything downstream is compromised.

In [ ]:
# torch.utils.data.Dataset supplies the interface Trainer expects for examples.
# SFTData eagerly encodes every row once; __len__ reports its size and __getitem__
# returns one encoded example. self.rows stores state on each dataset instance.
# The collator makes different-length examples into equally shaped batch tensors.
from torch.utils.data import Dataset
class SFTData(Dataset):
    def __init__(self, rows): self.rows=[encode(r) for r in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

# Choose the longest sequence in this batch, not the global maximum length.
# Dynamic padding saves work when all examples in a batch are short.
def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids":[], "attention_mask":[], "labels":[]}
    for b in batch:
        # k is this row's padding count. List concatenation appends [pad]*k input IDs,
        # [0]*k attention entries, and [-100]*k ignored loss labels. Actual answer EOS IDs
        # retain their normal labels even when pad_id and eos_id are the same number.
        k = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad]*k)
        out["attention_mask"].append(b["attention_mask"] + [0]*k)
        out["labels"].append(b["labels"] + [-100]*k)
    # A dictionary comprehension turns each list of padded rows into a tensor shaped
    # [batch_size, longest_sequence]. Here k means a dictionary key, independently of
    # the earlier padding-count variable inside the function.
    return {k: torch.tensor(v) for k, v in out.items()}

# Prepare separate datasets. This tokenizes the data but performs no model
# forward pass and does not update weights.
train_ds, val_ds = SFTData(train), SFTData(val)
print("ready:", len(train_ds), len(val_ds))

In [ ]:
# Attach LoRA and run supervised fine-tuning; this cell actually trains the model.
# PEFT provides LoraConfig/get_peft_model to add small trainable low-rank matrices
# while freezing base weights. Transformers supplies TrainingArguments and Trainer
# to manage batching, backpropagation, optimizer updates, logging, and validation.
# Stored training outputs are kept with this cell, so it has not been split.
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
lc, c = CFG["training"]["lora"], CFG["training"]["sft"]
# r sets adapter rank; alpha supplies standard alpha/r scaling; dropout regularizes
# adapter inputs. target_modules chooses projection layers and CAUSAL_LM selects
# next-token training. print_trainable_parameters confirms the actual trainable share.
m = get_peft_model(model, LoraConfig(r=lc["r"], lora_alpha=lc["alpha"], lora_dropout=lc["dropout"],
                                     target_modules=lc["target_modules"], task_type="CAUSAL_LM"))
m.print_trainable_parameters()

# Per-device batch size is the microbatch; gradient_accumulation_steps combines
# microbatch gradients before an optimizer update. Epochs repeat the training rows.
# eval_strategy runs validation each epoch; save_strategy='no' leaves adapter saving
# to the next cell. report_to=[] avoids external trackers. The seed controls supported
# randomness; reproducibility can still vary across hardware/library versions.
args = TrainingArguments(output_dir=str(lab.RUNS/"sft"), num_train_epochs=c["epochs"],
    per_device_train_batch_size=c["batch_size"], gradient_accumulation_steps=c["grad_accum"],
    learning_rate=c["lr"], logging_steps=5, eval_strategy="epoch", save_strategy="no",
    report_to=[], seed=CFG["seeds"]["train"])
# data_collator=collate applies our dynamic padding and preserves the -100 masks.
# Trainer/model compute loss only on supervised targets; validation loss uses the
# held-back SFT rows and does not itself measure dev-question accuracy.
trainer = Trainer(model=m, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate)
# Start gradient updates now. res.metrics reports training statistics, not a
# proof that all generated answers are grounded or that new facts generalize.
res = trainer.train()
print(res.metrics)

In [ ]:
# Save the trained LoRA adapter to adapters/sft and record its run metrics.
# PEFT's save_pretrained writes adapter weights/configuration, not the entire base
# checkpoint. Evaluation must reload this adapter with its compatible base model.
# Rerunning these saves updates the existing adapter directory and named run file.
m.save_pretrained(str(lab.ADAPTERS/"sft"))
lab.save_run("07_sft", {"metrics": res.metrics, "n_train": len(train)})
print("saved adapters/sft")

## Evaluate on the development set

Same scoring, same questions and the same ENGINEERED prompt as notebook 02, so the number is comparable.

If you restarted the kernel since training, run the setup and model-loading cells, then reload the saved adapter before evaluating: `from peft import PeftModel; m = PeftModel.from_pretrained(model, str(lab.ADAPTERS / "sft"))`.

In [ ]:
# The evaluation prompt is notebook 02's ENGINEERED instruction, byte-for-byte, with no
# documents, so the before/after comparison is fair. (SFT itself trained with the shorter
# SYSTEM prompt defined in the masking cell above.)
ENGINEERED = (
    "You are the customer assistant for Meridian Motors, an electric two-wheeler company. "
    "Answer only from Meridian's official documents. If the documents do not contain the "
    "answer, say plainly that you do not have that information and offer to connect the "
    "customer to a dealer. Never guess a number. Never round a range figure upward."
)

In [ ]:
# Evaluate the trained adapter with notebook 02's settings: ENGINEERED prompt, no documents,
# greedy decoding, the configured new-token cap and a 3,072-token input cap.
from tqdm.auto import tqdm
fidx, dev = lab.fact_index(), lab.eval_set("dev")
MAX_NEW = CFG["inference"]["max_new_tokens"]

# Trainer leaves the model in training mode. eval() switches dropout off. Always do this
# before generating, or answers come from a noisier model than the one you saved.
m.eval()
assert not m.training

def ask(q):
    t = tok.apply_chat_template([{"role": "system", "content": ENGINEERED}, {"role": "user", "content": q}],
                                tokenize=False, add_generation_prompt=True)
    i = tok(t, return_tensors="pt", truncation=True, max_length=3072).to(DEVICE)
    with torch.no_grad():
        o = m.generate(**i, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
    # Slice away the prompt tokens, then decode only the answer.
    return tok.decode(o[0][i["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Keep heuristic scores and raw answers separately. The shared scorer matches numbers and
# refusal keywords; it can misjudge colour lists and Arc 110's fast-charge value 0.
rows, raw = [], []
for q in tqdm(dev, desc="sft eval"):
    a = ask(q["question"]); rows.append(lab.score_answer(a, q, fidx))
    raw.append({"id": q["id"], "question": q["question"], "answer": a, "answerable": q["answerable"]})
s = lab.summarise(rows)
lab.save_run("08_sft_eval", {"summary": s, "answers": raw, "condition": "engineered prompt, no documents",
                             "system_prompt": ENGINEERED, "max_new_tokens": MAX_NEW})
print(json.dumps(s, indent=2))

## What to notice

Compare against notebook 02. Typically SFT **improves format and refusal phrasing markedly** while doing less for factual accuracy than people expect. That split is the lesson: SFT teaches behaviour, retrieval supplies facts.

Look especially at the `appropriate_refusal_rate`. If it rose, the refusal examples did their job.

**Next:** notebook 06, DPO. This is the one you came for.